# Prophet（2017）

- **核心原理**：加法分解＋節假日效應
- **變體**：NeuralProphet（加上自回歸與神經網路）
- **推薦資料集**：Peyton Manning（美式足球四分衛）的維基百科每日瀏覽量（Prophet 官方範例資料，2007-12～2016-01，已取 log）
- **實作**：🧰 套件（`prophet`）；另外用 NumPy 手寫一個「迷你 Prophet」（加權 ridge 回歸），理解它到底在做什麼
- **文獻**：Taylor & Letham (2018), *Forecasting at Scale*

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

Prophet 的模型：
$$y(t) = \underbrace{g(t)}_{\text{分段線性的趨勢}} + \underbrace{s(t)}_{\text{季節（傅立葉級數）}} + \underbrace{h(t)}_{\text{節日效果}} + \varepsilon_t$$

和前幾章的差別：

| | 第 1、2、6 章 | 本章 Prophet |
| --- | --- | --- |
| 看待時間的方式 | **遞迴**：今天由昨天推出來（狀態、過去的值） | **曲線配適**：$y$ 是「時間 $t$」的函數（一個廣義加法模型，GAM）—— 不看過去的 $y$ |
| 季節 | 固定週期 $m$，一種季節 | **多種季節**一起（週、年），用傅立葉級數 → 不需要等間隔、可以有缺漏的日期 |
| 趨勢的轉折 | 平滑地追蹤 | **轉折點**（changepoints）：趨勢的斜率在某些日期改變，用先驗（懲罰）控制轉折的多寡 |
| 特殊事件 | — | 節日／事件清單（季後賽、超級盃） |
| 估計 | 遞迴 + 最佳化 | 對所有參數**一次**做最大後驗（MAP）—— 本質是一個加了懲罰的線性回歸 |

一句話：**Prophet = 一個特徵設計好的線性回歸：「時間」被展開成「轉折點的 hinge 特徵 + 週、年的 sin／cos + 節日的 0/1」，再加上懲罰避免過擬合。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_peyton_manning()` 回傳 DataFrame（欄位 `ds` 日期、`y` = log(每日瀏覽量)）—— Prophet 要求的格式就是這兩欄。共 2,905 天（中間缺 59 天）。**最後 365 個日曆天**（2015-01-21 之後；其中 2 天缺資料，所以是 363 筆）當測試集，往前是 train。所有指標都在 **log 尺度**上算（`y` 本來就是 log）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $t$ | 時間，縮放到 train 的 0～1 | |
| $g(t)$ | 趨勢：$k t + m + \sum_j \delta_j (t - c_j)_+$ | |
| $c_j$、$\delta_j$ | 第 $j$ 個轉折點的位置、斜率的改變量 | 25 個，放在 train 的前 80% |
| $s(t)$ | 季節：$\sum_{n=1}^{N}\big[a_n\cos\frac{2\pi n d}{P} + b_n\sin\frac{2\pi n d}{P}\big]$ | 年：$P = 365.25$、$N = 10$；週：$P = 7$、$N = 3$ |
| $h(t)$ | 節日：每個（節日, 偏移天數）一個 0/1 特徵 | 季後賽、超級盃，當天與隔天 |
| $X$、$\beta$ | 把上面全部特徵排成的設計矩陣、係數 | |
| $\lambda$ | 各組特徵的懲罰強度（對應 Prophet 的 prior scale：$\lambda$ 大 ↔ prior scale 小） | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第一部分第 2 章 | 線性回歸 | Ridge：$\hat\beta = (X^\top X + \Lambda)^{-1}X^\top y$，$\Lambda$ 是對角的懲罰；懲罰 = 高斯先驗的最大後驗（MAP）。Lasso（L1）= Laplace 先驗，會讓係數剛好變 0。 |
| 第 1 章 | 指數平滑 | 季節性 naive、MASE（`season` = 週期）；遞迴的模型「最近的資料權重大」。 |
| 第 6 章 | STL | 加法分解 $T + S + R$；趨勢用 LOESS（局部）、季節用每個月的子序列。Prophet 的趨勢是**全域**的分段直線，季節是**全域**的傅立葉級數。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 傅立葉級數**：週期 $P$ 的任何（平滑的）函數 $\approx \sum_{n=1}^N [a_n\cos(2\pi n d/P) + b_n\sin(2\pi n d/P)]$。$N$ 大 → 可以表示越細的形狀（但也越容易過擬合）。例：$P = 4$、$n = 1$、$d = 0, 1, 2, 3$ → $\sin = (0, 1, 0, -1)$、$\cos = (1, 0, -1, 0)$。

**2. hinge 函數** $(t - c)_+ = \max(t - c, 0)$：在 $c$ 之前是 0、之後以斜率 1 增加。$g(t) = kt + m + \sum_j\delta_j(t - c_j)_+$ → 過了 $c_j$ 之後，斜率從 $k$ 變成 $k + \delta_j$（分段直線、連續）。

**3. 懲罰回歸（Ridge）**：$\min\lVert y - X\beta\rVert^2 + \sum_i\lambda_i\beta_i^2$ → $\hat\beta = (X^\top X + \text{diag}(\lambda))^{-1}X^\top y$。每一組特徵可以有不同的 $\lambda$（不懲罰的設 0，例如截距）。例：$X = \begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$、$y = (1, 2, 3)$、$\lambda = (0, 0)$ → $\beta = (1, 1)$；$\lambda = (0, 10^6)$（斜率被壓成 0）→ $\beta \approx (2, 0)$。

**4. MAP 與先驗**：$\beta_i \sim N(0, \tau_i^2)$ 的先驗 + 高斯誤差 $\sigma^2$ → MAP = Ridge，$\lambda_i = \sigma^2/\tau_i^2$。Prophet 的 `changepoint_prior_scale` 就是轉折點係數 $\delta$ 的先驗尺度（Laplace 分布，類似 L1）—— 越小，轉折越少、趨勢越硬。

#### 程式

**5. 日期運算**：`(ds − t0).dt.days` → 天數；`pd.Timedelta(days=k)`。
**6. `np.isin(dates, targets)`**：每個日期是否在清單中。
**7. Prophet 的 API**：`Prophet(...).fit(df)`（`df` 有 `ds`、`y`）→ `m.make_future_dataframe(periods=h)` 或自己給 `ds` → `m.predict(df_future)` 回傳 `yhat`、`yhat_lower`、`yhat_upper`、`trend`、`weekly`、`yearly`、`holidays` 等欄。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_peyton_manning

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("cmdstanpy").disabled = True  # Prophet 底層（Stan）的訊息
logging.getLogger("prophet").setLevel(logging.ERROR)

df = load_peyton_manning()
df = df.assign(ds=pd.to_datetime(df["ds"]), y=df["y"].astype(float))[["ds", "y"]].sort_values("ds").reset_index(drop=True)
TEST_START = df["ds"].max() - pd.Timedelta(days=364)
train, test = df[df["ds"] < TEST_START].reset_index(drop=True), df[df["ds"] >= TEST_START].reset_index(drop=True)
full_days = pd.date_range(df["ds"].min(), df["ds"].max(), freq="D")
print("全部：", len(df), "天", df["ds"].min().date(), "～", df["ds"].max().date(), "；缺的日期：", len(full_days) - len(df), "天")
print("train：", len(train), "天，到", train["ds"].max().date(), "；test：", len(test), "天，從", test["ds"].min().date())

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(train["ds"], train["y"], lw=0.5, label="train")
ax.plot(test["ds"], test["y"], lw=0.5, label="test")
ax.set_ylabel("log(daily page views)")
ax.legend()
plt.show()

In [ ]:
# 週、年的季節性；最高的日子
# 觀察：
#   - 星期幾最高、最低？差多少（log 的差 0.1 ≈ 10%）？
#   - 一年裡哪幾個月最高？（美式足球的賽季：9 月～隔年 2 月）
#   - 瀏覽量最高的 10 天是哪些日期？它們有什麼共同點？
tr = train.assign(dow=train["ds"].dt.day_of_week, month=train["ds"].dt.month)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6))
sns.boxplot(data=tr, x="dow", y="y", ax=axes[0])
axes[0].set_xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
sns.boxplot(data=tr, x="month", y="y", ax=axes[1])
plt.show()
print("星期幾的中位數：", tr.groupby("dow")["y"].median().round(3).to_dict())
display(train.nlargest(10, "y").assign(weekday=lambda d: d["ds"].dt.day_name()))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 趨勢：2008 → 2015 的水準怎麼變？有沒有「轉折」（斜率改變）的時間點？（提示：Peyton Manning 2011 年因傷整季沒出賽、2012 年轉隊到丹佛野馬）
# (2) 週季節：哪一天最高？為什麼？（NFL 大多在星期日比賽）
# (3) 年季節：賽季（9 月～隔年 2 月）與休賽季差多少？
# (4) 最高的 10 天：多半是什麼樣的日子？如果一個模型「只有趨勢 + 週 + 年的季節」，這些日子會被低估還是高估？
# (5) 中間缺了 59 天：第 1～2 章的方法（指數平滑、ARIMA）遇到缺漏的日期要怎麼處理？Prophet 為什麼不在乎？
# 答：

---
## 2. Baseline（預測 test 的 365 天）

| Baseline | 預測 |
| --- | --- |
| naive | train 的最後一天 |
| 最近 28 天平均 | |
| 週季節性 naive | 最後一週的同一天 |
| **去年同一天** | 364 天前（同一個星期幾）；缺的那天用最近的前一天 |

**指標**（`forecast_metrics`，log 尺度）：MAE、RMSE、MASE（`season=7`：分母是 train 上「上週同一天」的一步誤差）。

In [ ]:
from learning_map.metrics import forecast_metrics

y_tr = train["y"].to_numpy()
H = len(test)
series_all = df.set_index("ds")["y"].reindex(full_days).ffill()  # 補缺的日期（只在 baseline 用）
base = {
    "naive": np.full(H, y_tr[-1]),
    "mean of last 28 days": np.full(H, y_tr[-28:].mean()),
    "weekly seasonal naive": train.set_index("ds")["y"].reindex(pd.date_range(train["ds"].max() - pd.Timedelta(days=6), train["ds"].max())).ffill().to_numpy()[
        ((test["ds"] - (train["ds"].max() - pd.Timedelta(days=6))).dt.days.to_numpy()) % 7],
    "same day last year (364 d)": series_all.reindex(test["ds"] - pd.Timedelta(days=364)).to_numpy(),
}
test_table = pd.DataFrame({k: forecast_metrics(test["y"], v, train["y"], season=7) for k, v in base.items()}).T
display(test_table[["mae", "rmse", "smape", "mase"]].round(4))

In [ ]:
# TODO 2（問答）
#
# (1) 哪一個 baseline 最好？「去年同一天」為什麼比 naive 好很多？
# (2) MASE 的分母是「上週同一天」的一步誤差。一年後的預測 MASE ≈ 1 代表什麼？（提示：365 天後的預測，跟「只預測下一週」一樣準）
# (3) 先寫下預測：Prophet 的 MASE 會比「去年同一天」好多少？
# 答：

---
## 3. 手寫模型：迷你 Prophet（NumPy）

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 季節的傅立葉特徵 | 3 |
| 3B | 趨勢的 hinge 特徵（轉折點） | 4 |
| 3C | 節日特徵 | 5 |
| 3D | 加權 Ridge（不同組不同懲罰）；對照 Prophet 的成分 | 6、7 |
| 3E | 實驗：轉折點懲罰、傅立葉階數、節日 | 8 |

**程式結構**：`MiniProphet` —— `_design(ds)` 把日期變成設計矩陣 $X$（並記下每一欄屬於哪一組：`trend`、`changepoint`、`yearly`、`weekly`、`holiday`），`fit(ds, y)` 解 Ridge，`predict(ds)` 回傳每一組的貢獻與總和（跟 Prophet 的 `trend`、`yearly`、`weekly`、`holidays`、`yhat` 對應）。

跟真正的 Prophet 的差別：Prophet 對轉折點用 **Laplace 先驗**（像 L1：大部分 $\delta_j$ 剛好是 0）、用 Stan 做 MAP、把 $y$ 先縮放到 0～1；迷你版全部用 **L2**、閉式解。結果會很接近但不會一模一樣。

### 3A. 傅立葉特徵

#### 符號表（3A–3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $d$ | 從 1970-01-01 起算的天數（Prophet 的慣例） | $(n,)$ | `days` |
| $P$、$N$ | 週期、階數 | | `period`、`order` |
| 傅立葉特徵 | $[\sin(2\pi\cdot 1 d/P), \cos(2\pi\cdot 1 d/P), \dots, \sin(2\pi N d/P), \cos(2\pi N d/P)]$ | $(n, 2N)$ | |
| $t$ | 縮放後的時間 $(ds - t_0)/T$ | $(n,)$ | `t` |
| $c$ | 轉折點的位置（$t$ 的尺度） | $(J,)$ | `self.cps_` |

**下標公式**：
- TODO 3：對 $n = 1..N$：`np.sin(2π n d / P)`、`np.cos(2π n d / P)`，依序 column_stack（先 sin 再 cos）。
- TODO 4：`np.column_stack([np.maximum(t − c, 0) for c in cps])`。
- TODO 5：對每個（節日名稱, 偏移 $k$）：`np.isin(ds, 節日日期 + k 天)` 轉成 0/1。

#### 最小例子：TODO 3、4、5

- TODO 3：$d = (0, 1, 2, 3)$、$P = 4$、$N = 1$ → `[[0, 1], [1, 0], [0, −1], [−1, 0]]`（sin、cos 兩欄）。
- TODO 4：$t = (0, 0.25, 0.5, 0.75, 1)$、轉折點 $c = 0.5$ → $(0, 0, 0, 0.25, 0.5)$。
- TODO 5：日期 2020-01-01～01-05、節日 2020-01-02、窗口 0～1 天 → 「+0」欄 $(0, 1, 0, 0, 0)$、「+1」欄 $(0, 0, 1, 0, 0)$。

In [ ]:
from typing import Self


def fourier_features(days: np.ndarray, period: float, order: int) -> np.ndarray:
    """(n, 2·order)：sin(2π·1·d/P), cos(2π·1·d/P), ..., sin(2π·N·d/P), cos(2π·N·d/P)。"""
    # TODO 3（程式）：見下標公式
    raise NotImplementedError("TODO 3：傅立葉特徵")


def changepoint_features(t: np.ndarray, cps: np.ndarray) -> np.ndarray:
    """(n, J)：每個轉折點一欄 max(t − c_j, 0)。"""
    # TODO 4（程式）
    raise NotImplementedError("TODO 4：hinge 特徵")


def holiday_features(ds: pd.Series, holidays: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    """holidays：Prophet 的格式（holiday、ds、lower_window、upper_window）。回傳 (n, 欄數) 的 0/1 與欄名（例如 "playoff_+1"）。"""
    cols, names = [], []
    for name, g in holidays.groupby("holiday", sort=True):
        for k in range(int(g["lower_window"].min()), int(g["upper_window"].max()) + 1):
            # TODO 5（程式）：這個節日的所有日期往後移 k 天，ds 落在其中的為 1
            raise NotImplementedError("TODO 5：節日特徵")
            names.append(f"{name}_{k:+d}")
    return np.column_stack(cols), names


# 寫完 TODO 3–5 之後，取消下面的註解來對照（見上面 markdown 的正確答案）：
# print(fourier_features(np.arange(4.0), 4, 1).round(4))
# print(changepoint_features(np.array([0, 0.25, 0.5, 0.75, 1.0]), np.array([0.5])).ravel())
# print(holiday_features(pd.Series(pd.date_range("2020-01-01", periods=5)), pd.DataFrame({"holiday": "h", "ds": pd.to_datetime(["2020-01-02"]), "lower_window": 0, "upper_window": 1})))

### 3D. 加權 Ridge、對照 Prophet

$$\hat\beta = (X^\top X + \text{diag}(\lambda))^{-1}X^\top y$$

每一欄的 $\lambda$ 依它屬於哪一組：截距、斜率 $k$ 不懲罰（0）；轉折點 `lam_cp`；季節 `lam_season`；節日 `lam_hol`。

**對照 Prophet**：同樣的節日、預設的轉折點（25 個、前 80%）、年季節 10 階、週季節 3 階。比較兩者在 test 上的預測與各成分（趨勢、年季節、週季節、節日）的形狀。

**下標公式**：
- TODO 6：`A = X.T @ X + np.diag(pen)`；`beta = np.linalg.solve(A, X.T @ y)`。

#### 最小例子：TODO 6

前置複習 3 的例子：$\lambda = (0, 0)$ → $(1, 1)$；$\lambda = (0, 10^6)$ → $\approx (2, 0)$。

In [ ]:
def ridge_solve(X: np.ndarray, y: np.ndarray, pen: np.ndarray) -> np.ndarray:
    """min ‖y − Xβ‖² + Σ pen_i β_i²。"""
    # TODO 6（程式）：見下標公式
    raise NotImplementedError("TODO 6：Ridge")


# 寫完 TODO 6 之後，取消下面的註解來對照（應為 [1. 1.] 與 [2. 0.]）：
# X_mini = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]]); y_mini = np.array([1.0, 2.0, 3.0]); print(ridge_solve(X_mini, y_mini, np.zeros(2)).round(4), ridge_solve(X_mini, y_mini, np.array([0.0, 1e6])).round(4))


class MiniProphet:
    """趨勢（分段線性）+ 年／週季節（傅立葉）+ 節日，加權 Ridge。"""

    def __init__(self, n_changepoints: int = 25, changepoint_range: float = 0.8, yearly_order: int = 10, weekly_order: int = 3,
                 holidays: pd.DataFrame | None = None, lam_cp: float = 1.0, lam_season: float = 0.1, lam_hol: float = 0.1) -> None:
        self.n_changepoints, self.changepoint_range = n_changepoints, changepoint_range
        self.yearly_order, self.weekly_order, self.holidays = yearly_order, weekly_order, holidays
        self.lam = {"trend": 0.0, "changepoint": lam_cp, "yearly": lam_season, "weekly": lam_season, "holiday": lam_hol}

    def _design(self, ds: pd.Series) -> tuple[np.ndarray, np.ndarray]:
        t = (ds - self.t0_).dt.days.to_numpy() / self.T_
        days = (ds - pd.Timestamp("1970-01-01")).dt.days.to_numpy().astype(float)
        blocks = [("trend", np.column_stack([np.ones(len(ds)), t])), ("changepoint", changepoint_features(t, self.cps_)),
                  ("yearly", fourier_features(days, 365.25, self.yearly_order)), ("weekly", fourier_features(days, 7.0, self.weekly_order))]
        if self.holidays is not None:
            blocks.append(("holiday", holiday_features(ds, self.holidays)[0]))
        X = np.column_stack([b for _, b in blocks])
        groups = np.concatenate([[g] * b.shape[1] for g, b in blocks])
        return X, groups

    def fit(self, ds: pd.Series, y: np.ndarray) -> Self:
        self.t0_ = ds.min()
        self.T_ = (ds.max() - self.t0_).days
        self.cps_ = np.linspace(0, self.changepoint_range, self.n_changepoints + 1)[1:]
        X, self.groups_ = self._design(ds)
        pen = np.array([self.lam[g] for g in self.groups_])
        self.beta_ = ridge_solve(X, np.asarray(y, dtype=float), pen)
        return self

    def predict(self, ds: pd.Series) -> pd.DataFrame:
        X, groups = self._design(ds)
        out = {"ds": ds.to_numpy()}
        for g in ["trend", "changepoint", "yearly", "weekly", "holiday"]:
            out[g] = X[:, groups == g] @ self.beta_[groups == g] if (groups == g).any() else np.zeros(len(ds))
        out = pd.DataFrame(out)
        out["trend"] = out["trend"] + out.pop("changepoint")  # 跟 Prophet 一樣：trend = 直線 + 轉折
        out["yhat"] = out[["trend", "yearly", "weekly", "holiday"]].sum(axis=1)
        return out

In [ ]:
from prophet import Prophet

playoffs = pd.DataFrame({"holiday": "playoff", "ds": pd.to_datetime(["2008-01-13", "2009-01-03", "2010-01-16", "2010-01-24", "2010-02-07", "2011-01-08", "2013-01-12",
                                                                    "2014-01-12", "2014-01-19", "2014-02-02", "2015-01-11", "2016-01-17", "2016-01-24", "2016-02-07"]),
                         "lower_window": 0, "upper_window": 1})
superbowls = pd.DataFrame({"holiday": "superbowl", "ds": pd.to_datetime(["2010-02-07", "2014-02-02", "2016-02-07"]), "lower_window": 0, "upper_window": 1})
HOLIDAYS = pd.concat([playoffs, superbowls], ignore_index=True)  # Prophet 官方教學的節日表

mini = MiniProphet(holidays=HOLIDAYS).fit(train["ds"], y_tr)
pred_mini = mini.predict(test["ds"])
t0 = time.perf_counter()
pm = Prophet(holidays=HOLIDAYS).fit(train)
fc_p = pm.predict(test[["ds"]])
print(f"Prophet fit + predict：{time.perf_counter() - t0:.1f} 秒")
cmp = pd.DataFrame({"mini Prophet (ridge)": forecast_metrics(test["y"], pred_mini["yhat"], train["y"], season=7),
                    "Prophet": forecast_metrics(test["y"], fc_p["yhat"], train["y"], season=7)}).T
display(cmp[["mae", "rmse", "mase"]].round(4))
print("兩者的 test 預測：相關係數", round(float(np.corrcoef(pred_mini["yhat"], fc_p["yhat"])[0, 1]), 4), "；最大差異", round(float(np.abs(pred_mini["yhat"] - fc_p["yhat"]).max()), 3))

pred_tr_mini = mini.predict(train["ds"])
fc_tr_p = pm.predict(train[["ds"]])
fig, axes = plt.subplots(2, 2, figsize=(16, 7))
axes[0, 0].plot(train["ds"], fc_tr_p["trend"], label="Prophet")
axes[0, 0].plot(train["ds"], pred_tr_mini["trend"], ls="--", label="mini")
axes[0, 0].set_title("trend")
one_year = train["ds"].between("2013-01-01", "2013-12-31")
axes[0, 1].plot(train["ds"][one_year], fc_tr_p["yearly"][one_year], label="Prophet")
axes[0, 1].plot(train["ds"][one_year], pred_tr_mini["yearly"][one_year], ls="--", label="mini")
axes[0, 1].set_title("yearly (2013)")
wk = train["ds"].between("2013-03-04", "2013-03-10")
axes[1, 0].plot(train["ds"][wk].dt.day_name().str[:3], fc_tr_p["weekly"][wk], "o-", label="Prophet")
axes[1, 0].plot(train["ds"][wk].dt.day_name().str[:3], pred_tr_mini["weekly"][wk], "o--", label="mini")
axes[1, 0].set_title("weekly")
axes[1, 1].plot(test["ds"], test["y"], color="k", lw=0.5, label="actual")
axes[1, 1].plot(test["ds"], fc_p["yhat"], label="Prophet")
axes[1, 1].plot(test["ds"], pred_mini["yhat"], ls="--", label="mini")
axes[1, 1].set_title("test forecast")
for ax in axes.ravel():
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# TODO 7（問答）TODO 3–6 與對照
#
# (1) 迷你 Prophet 跟 Prophet 的 test MASE 差多少？預測的相關係數？哪一個成分（趨勢、年、週）差最多？為什麼？
#     （提示：轉折點的先驗 —— L2 讓每個 δ 都一點點，Laplace 讓少數幾個 δ 很大、其他是 0）
# (2) 設計矩陣有幾欄？（2 + 25 + 20 + 6 + 4）跟 train 的天數比，參數算多嗎？為什麼還需要懲罰？
# (3) 迷你 Prophet 在 train 上能不能「完美」配適 Super Bowl 那幾天？跟沒有節日特徵時比呢？
# 答：

### 3E. 實驗（迷你 Prophet，看 test 只為了理解；選模型在第 5 節用 CV）

| 實驗 | 問題 |
| --- | --- |
| 1 | 轉折點懲罰 `lam_cp`：0.01、1、100、$10^4$ —— 趨勢的形狀與預測怎麼變？ |
| 2 | 年季節的傅立葉階數：2、10、30 |
| 3 | 拿掉節日特徵：季後賽、超級盃那幾天的殘差變多大？ |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
rows = []
for lam_cp in [0.01, 1.0, 100.0, 1e4]:
    m_ = MiniProphet(holidays=HOLIDAYS, lam_cp=lam_cp).fit(train["ds"], y_tr)
    tr_all = m_.predict(pd.concat([train["ds"], test["ds"]], ignore_index=True))
    axes[0].plot(tr_all["ds"], tr_all["trend"], label=f"lam_cp = {lam_cp:g}")
    rows.append({"experiment": f"lam_cp = {lam_cp:g}", **forecast_metrics(test["y"], m_.predict(test["ds"])["yhat"], train["y"], season=7)})
axes[0].axvline(TEST_START, color="k", lw=0.5)
axes[0].legend(fontsize=8)
axes[0].set_title("trend (train + test period)")
for order in [2, 10, 30]:
    m_ = MiniProphet(holidays=HOLIDAYS, yearly_order=order).fit(train["ds"], y_tr)
    p_ = m_.predict(train["ds"])
    axes[1].plot(train["ds"][one_year], p_["yearly"][one_year], label=f"yearly order {order}")
    rows.append({"experiment": f"yearly order {order}", **forecast_metrics(test["y"], m_.predict(test["ds"])["yhat"], train["y"], season=7)})
axes[1].legend(fontsize=8)
axes[1].set_title("yearly component (2013)")
plt.show()

no_hol = MiniProphet(holidays=None).fit(train["ds"], y_tr)
rows.append({"experiment": "no holidays", **forecast_metrics(test["y"], no_hol.predict(test["ds"])["yhat"], train["y"], season=7)})
display(pd.DataFrame(rows).set_index("experiment")[["mae", "rmse", "mase"]].round(4))
hol_days = train["ds"].isin(pd.concat([HOLIDAYS["ds"], HOLIDAYS["ds"] + pd.Timedelta(days=1)]))
for name, m_ in [("with holidays", mini), ("no holidays", no_hol)]:
    res = y_tr - m_.predict(train["ds"])["yhat"].to_numpy()
    print(f"{name:14s}：節日（與隔天）的平均殘差 {res[hol_days].mean():+.3f}；其他日子 {res[~hol_days].mean():+.3f}")
print("學到的節日效果（log）：", dict(zip(holiday_features(train["ds"], HOLIDAYS)[1], mini.beta_[mini.groups_ == "holiday"].round(3))))

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 轉折點懲罰：lam_cp 很小（0.01）時趨勢長什麼樣子？很大（1e4）時呢？哪一個在 test 最好？「轉折點只放在前 80%」的理由是什麼？
#     （提示：最後 20% 的資料太少，估出來的轉折很不可靠 —— 而且會被直接外插到未來）
# (2) 傅立葉階數：order 2 的年季節是什麼形狀？30 呢？太大的 order 會出現什麼問題？
# (3) 節日：學到的「季後賽當天」效果大約是 log 多少？換成倍數是幾倍（exp）？超級盃呢？沒有節日特徵時，這些日子的殘差平均多大？
# 答：

---
## 4. 套件實作：`prophet`

| 參數 | 預設 | 意思 |
| --- | --- | --- |
| `changepoint_prior_scale` | 0.05 | 轉折點 $\delta$ 的 Laplace 先驗尺度：大 → 趨勢更有彈性 |
| `seasonality_prior_scale` | 10 | 季節係數的常態先驗尺度 |
| `holidays_prior_scale` | 10 | 節日係數 |
| `seasonality_mode` | `"additive"` | 或 `"multiplicative"` |
| `n_changepoints`、`changepoint_range` | 25、0.8 | |
| `interval_width` | 0.8 | 預測區間（用模擬趨勢未來的轉折 + 殘差） |

方法：`m.plot_components(forecast)`、`m.changepoints`、`from prophet.plot import add_changepoints_to_plot`。

In [ ]:
from prophet.plot import add_changepoints_to_plot

fig = pm.plot(fc_p, figsize=(14, 4))
add_changepoints_to_plot(fig.gca(), pm, fc_p)
plt.show()
pm.plot_components(pm.predict(df[["ds"]]), figsize=(14, 9))
plt.show()
deltas = pm.params["delta"].ravel()
print(f"25 個轉折點中，|δ| > 0.01 的有 {(np.abs(deltas) > 0.01).sum()} 個（Laplace 先驗 → 大部分接近 0）")
inside = ((test["y"] >= fc_p["yhat_lower"]) & (test["y"] <= fc_p["yhat_upper"])).mean()
print(f"80% 預測區間涵蓋 test 的 {inside:.1%}")

In [ ]:
# TODO 9（問答）
#
# (1) 圖上紅色的轉折點有幾個「真的有轉」？對應到哪些事件（受傷、轉隊、退休傳聞）？
# (2) 80% 預測區間涵蓋了 test 的幾 %？Prophet 的區間來源有兩個：殘差的雜訊、以及「未來可能出現的轉折」—— 哪一個讓區間越往後越寬？
# (3) plot_components 的週季節、年季節跟 EDA 一致嗎？
# 答：

---
## 5. 調參：Prophet 的滾動起點 CV

`prophet.diagnostics.cross_validation(m, initial, period, horizon)`：從 `initial` 天之後開始，每隔 `period` 天設一個起點（cutoff），每個起點用之前的資料重新 fit、預測接下來 `horizon` 天。這裡用 `initial="1095 days"`（3 年）、`period="180 days"`、`horizon="365 days"`，**只在 train 上做**。

網格：`changepoint_prior_scale` ∈ {0.01, 0.05, 0.5} × `seasonality_prior_scale` ∈ {1, 10}，共 6 組；比較所有起點的平均 MAE（log 尺度）。也把迷你 Prophet（3 個 `lam_cp`）放進同樣的起點比較。

In [ ]:
from prophet.diagnostics import cross_validation

t0 = time.perf_counter()
grid_rows = []
cutoffs = None
for cps, sps in [(c, s) for c in (0.01, 0.05, 0.5) for s in (1.0, 10.0)]:
    m_ = Prophet(holidays=HOLIDAYS, changepoint_prior_scale=cps, seasonality_prior_scale=sps).fit(train)
    cv = cross_validation(m_, initial="1095 days", period="180 days", horizon="365 days", disable_tqdm=True)
    cutoffs = sorted(cv["cutoff"].unique())
    grid_rows.append({"model": f"Prophet cps={cps}, sps={sps}", "cv MAE": float(np.abs(cv["y"] - cv["yhat"]).mean())})
print(f"Prophet 網格：{time.perf_counter() - t0:.0f} 秒；{len(cutoffs)} 個起點：", [str(pd.Timestamp(c).date()) for c in cutoffs])

for lam_cp in [0.1, 1.0, 100.0]:
    errs = []
    for c in cutoffs:
        c = pd.Timestamp(c)
        tr_c = train[train["ds"] <= c]
        va_c = train[(train["ds"] > c) & (train["ds"] <= c + pd.Timedelta(days=365))]
        p_ = MiniProphet(holidays=HOLIDAYS, lam_cp=lam_cp).fit(tr_c["ds"], tr_c["y"].to_numpy()).predict(va_c["ds"])
        errs.append(np.abs(va_c["y"].to_numpy() - p_["yhat"].to_numpy()))
    grid_rows.append({"model": f"mini Prophet lam_cp={lam_cp:g}", "cv MAE": float(np.concatenate(errs).mean())})
grid = pd.DataFrame(grid_rows).set_index("model").sort_values("cv MAE")
display(grid.round(4))
best_name = grid.index[0]
print("CV MAE 最低：", best_name)

In [ ]:
# TODO 10（問答）
#
# (1) 6 組 Prophet 設定的 CV MAE 差多少？changepoint_prior_scale 的影響大，還是 seasonality_prior_scale？
# (2) 迷你 Prophet 跟 Prophet 在同樣的起點上比如何？
# (3) Prophet 的 CV 每個起點都要重新 fit。如果每次 fit 要 1 分鐘、資料有 1,000 條序列（「Forecasting at Scale」），網格搜尋要多久？你會怎麼取捨？
# 答：

---
## 6. 測試集評估

用 CV 選出的設定在整個 train 上 fit，預測 test 的 365 天，只評估一次。也放進預設的 Prophet、迷你 Prophet、最好的 baseline。成績寫進 `results/peyton_manning.csv`。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "peyton_manning"
if best_name.startswith("Prophet"):
    cps_b = float(best_name.split("cps=")[1].split(",")[0])
    sps_b = float(best_name.split("sps=")[1])
    final = Prophet(holidays=HOLIDAYS, changepoint_prior_scale=cps_b, seasonality_prior_scale=sps_b).fit(train)
    fc_final = final.predict(test[["ds"]])["yhat"].to_numpy()
    params = {"changepoint_prior_scale": cps_b, "seasonality_prior_scale": sps_b}
else:
    lam_b = float(best_name.split("lam_cp=")[1])
    fc_final = MiniProphet(holidays=HOLIDAYS, lam_cp=lam_b).fit(train["ds"], y_tr).predict(test["ds"])["yhat"].to_numpy()
    params = {"lam_cp": lam_b}
eval_models = {
    "same day last year (364 d)": (base["same day last year (364 d)"], {}),
    "mini Prophet (ridge, scratch)": (pred_mini["yhat"].to_numpy(), {"lam_cp": 1.0}),
    "Prophet default + holidays": (fc_p["yhat"].to_numpy(), {}),
    f"{best_name} (CV-selected)": (fc_final, params),
}
for name, (fc, prm) in eval_models.items():
    test_table.loc[name] = forecast_metrics(test["y"], fc, train["y"], season=7)
    log_result(BENCH, name, test_table.loc[name].dropna().to_dict(), horizon=H, **prm)
display(test_table[["mae", "rmse", "smape", "mase"]].round(4))
compare(BENCH, "mase", higher_is_better=False)

---
## 7. 診斷

In [ ]:
res_test = test["y"].to_numpy() - fc_final
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
axes[0].plot(test["ds"], res_test, lw=0.6)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_title("test residual (log)")
pd.Series(res_test).groupby(test["ds"].dt.day_of_week.to_numpy()).mean().plot.bar(ax=axes[1])
axes[1].set_title("mean residual by weekday")
pd.Series(res_test).groupby(test["ds"].dt.month.to_numpy()).mean().plot.bar(ax=axes[2])
axes[2].set_title("mean residual by month")
plt.show()
worst = np.argsort(-np.abs(res_test))[:8]
display(pd.DataFrame({"ds": test["ds"].iloc[worst].dt.date.to_numpy(), "actual": test["y"].iloc[worst].round(2).to_numpy(), "forecast": fc_final[worst].round(2), "residual": res_test[worst].round(2)}))
print("test 期間的節日：", HOLIDAYS[HOLIDAYS["ds"] >= TEST_START][["holiday", "ds"]].assign(ds=lambda d: d["ds"].dt.date).to_dict("records"))

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：Prophet 比「去年同一天」好多少？CV 選的設定跟預設的比呢？迷你 Prophet 呢？
# (2) 殘差：整體偏高還是偏低？哪些月份誤差最大？誤差最大的幾天是什麼日子？
# (3) 節日表的「資訊洩漏」：HOLIDAYS 裡 2015、2016 年的季後賽日期，是「Peyton Manning 的球隊有打進季後賽」的那幾天 ——
#     在 2015-01-20（train 的最後一天）能事先知道嗎？如果不能，test 的成績有沒有被高估？你會怎麼做才公平？
# (4) Prophet 的適用範圍：它不看「過去的 y」（沒有自回歸）。對哪一種序列特別好（強季節、多種週期、有已知的事件）？哪一種不好（短期的動態很重要，例如股價、第 4 章的波動率）？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Peyton Manning 維基百科瀏覽量（log；test = 最後 365 天）

| 方法 | 做法 | CV MAE | test MAE | test MASE |
| --- | --- | --- | --- | --- |
| 去年同一天 | | — | | |
| 迷你 Prophet（Ridge） | | | | |
| Prophet（預設 + 節日） | | | | |
| Prophet（CV 選的） | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ Prophet 的完整模型：logistic 成長（有上限的趨勢）、乘法季節、外生變數（`add_regressor`）
- ☐ Laplace 先驗與 L1、MAP vs 完整貝氏（`mcmc_samples`）
- ☐ 預測區間的模擬方法

**延伸**
- ☐ NeuralProphet（加上 AR 與神經網路）
- ☐ 用 `add_country_holidays` 加國定假日；自訂季節（例如每月）
- ☐ 第 8 章起的機器學習／深度學習方法 vs Prophet（ETTh1 共用資料集）